# TabFM Evaluation

TabFM is Google's zero-shot foundation model for tabular data (released June
2026). It does not train: `.fit()` stores the training rows as context and
`.predict()` runs a single forward pass. There are no hyperparameters to tune
and no feature selection step, which is the model's main selling point.

This notebook evaluates it against the four existing models. Because a single
train/test split of a one-year series gives unstable results (see
`03_split_strategy.ipynb`), every comparison here is repeated across several
splits and reported with a confidence interval.

**Scope.** The evaluation is limited to two categories and five seeds. The
reason is measured, not arbitrary — see the runtime section below.

In [2]:
import sys, os
print(sys.executable)   # doit finir par Program Files\Python311\python.exe
print(os.getcwd())      # doit être C:\Users\user\internship

C:\Program Files\Python311\python.exe
C:\Users\user\internship\notebooks


In [3]:
import os, time
import pandas as pd

from src import config, features as feat, splits as spl, models as mdl
from src import validation as val
from src.metrics import evaluate_forecast

daily = pd.read_csv(config.DAILY_CLEAN, parse_dates=["Date"])
daily = daily[~daily["Category"].isin(config.EXCLUDED_CATEGORIES)]

print(daily.shape, "|", daily["Category"].nunique(), "categories")

(20196, 5) | 54 categories


## 1. Runtime cost

TabFM ships 6.6 GB of weights (regression head only; the full repository is
13 GB) and requires PyTorch. Two settings matter on CPU:

- `dtype=None` loads float32. The default bfloat16 is designed for GPU and is
  emulated on CPUs without native support.
- `torch.set_num_threads(os.cpu_count())` — PyTorch defaulted to 4 of 8 cores.

`n_estimators` controls how many forward passes are ensembled. The library
default is 32; this notebook uses 8, and the section below checks that the
reduction is small relative to the seed-to-seed noise.

In [ ]:
N_ESTIMATORS = 2
N_SEEDS = 3
CATEGORIES = ["Tealight Holders & Sets", "Jewellery - Earrings"]

t0 = time.time()
ok = mdl.enable_tabfm(n_estimators=N_ESTIMATORS)
print(f"load: {time.time() - t0:.1f}s")
print("models:", mdl.model_names())

if not ok:
    raise RuntimeError("TabFM unavailable - check installation")

TabFM registered.
load: 0.0s
models: ['7-Day Rolling Sum Baseline', 'Linear Regression', 'Random Forest', 'XGBoost', 'TabFM']


In [ ]:
# Time a single call, for the cost discussion in the report.
CAT = CATEGORIES[0]
X, Y = 7, 7

data = feat.build_features(daily, CAT, X=X, Y=Y)
target = config.target_name(X, Y)
features = feat.available_features(data)
splits = spl.stratified_block_split(data, target, random_state=42)

timings = {}
for name in mdl.model_names():
    t0 = time.time()
    mdl.MODELS[name](splits["train"], splits["validation"], splits["test"],
                     features, target)
    timings[name] = round(time.time() - t0, 2)

pd.Series(timings, name="seconds per fit+predict").sort_values().to_frame()

,seconds per fit+predict
7-Day Rolling Sum Baseline,0.00
Linear Regression,0.05
XGBoost,0.68
Random Forest,0.87
TabFM,1675.45


The gap in the table above is the practical finding: a Random Forest fits 238
rows in a fraction of a second, while TabFM takes orders of magnitude longer
per call. Extrapolated to 54 categories with 30 seeds, a full comparison would
run for hours on this machine. That is why the evaluation below is deliberately
scoped to two categories.

The two chosen categories bracket the difficulty range: **Tealight Holders &
Sets** is high-volume and regular, **Jewellery - Earrings** is sparse with
roughly 60% zero-demand days and is where every other model performs worst.

## 2. Comparison across repeated splits

`repeated_evaluation` re-draws the stratified block split `N_SEEDS` times and
evaluates all five models on each draw. `win_rate` is the fraction of draws a
model came first — a model winning 40% of draws is not "the best model", it is
the one that happened to win a single draw.

Note that TabFM's train metrics are NaN by design: it predicts rows that sit
inside its own context, so a train score would measure recall rather than fit.

In [ ]:
frames = []
for cat in CATEGORIES:
    t0 = time.time()
    frames.append(val.repeated_evaluation(daily, cat, X=X, Y=Y,
                                          seeds=range(N_SEEDS), stratified=True))
    print(f"{cat}: {time.time() - t0:.0f}s")

rep = pd.concat(frames, ignore_index=True)
rep.to_csv(config.RESULTS / "tabfm_comparison_raw.csv", index=False)

summary = val.summarise_repeats(rep)
summary.to_csv(config.RESULTS / "tabfm_comparison_summary.csv", index=False)
summary

Tealight Holders & Sets: 25544s
Jewellery - Earrings: 302s


,Category,Model,wape_mean,wape_std,wape_min,wape_max,r2_mean,win_rate
4,Jewellery - Earrings,XGBoost,71.307,4.558,66.098,74.564,0.258,0.333
3,Jewellery - Earrings,TabFM,71.328,12.813,56.916,81.433,0.530,0.333
2,Jewellery - Earrings,Random Forest,76.024,6.668,70.760,83.522,0.267,0.333
1,Jewellery - Earrings,Linear Regression,95.134,5.131,89.373,99.211,0.007,0.000
0,Jewellery - Earrings,7-Day Rolling Sum Baseline,118.743,14.528,105.290,134.149,-0.327,0.000
8,Tealight Holders & Sets,TabFM,20.292,1.910,19.144,22.497,0.619,0.667
5,Tealight Holders & Sets,7-Day Rolling Sum Baseline,24.189,8.365,14.571,29.765,0.410,0.333
9,Tealight Holders & Sets,XGBoost,24.636,6.108,17.599,28.567,0.451,0.000
7,Tealight Holders & Sets,Random Forest,25.199,5.717,18.613,28.876,0.412,0.000
6,Tealight Holders & Sets,Linear Regression,29.108,3.406,25.572,32.367,0.217,0.000


## 3. Is the difference real?

`is_difference_meaningful` compares two models on the *same* splits. The
pairing matters: both models see identical data on each seed, so the per-seed
difference cancels the split-to-split variation that dominates the absolute
numbers.

A negative `mean_diff` means TabFM is better. If the confidence interval
crosses zero, the two models are indistinguishable and no claim should be made
either way.

With only five seeds these intervals are wide. Treat a narrow result as
suggestive rather than established.

In [ ]:
rows = []
for cat in CATEGORIES:
    sub = rep[rep["Category"] == cat]
    for other in ["Random Forest", "XGBoost", "Linear Regression",
                  "7-Day Rolling Sum Baseline"]:
        d = val.is_difference_meaningful(sub, "TabFM", other)
        if d.get("n", 0) < 2:
            continue
        decided = (d["ci_low"] < 0) == (d["ci_high"] < 0)
        rows.append({
            "Category": cat,
            "vs": other,
            "mean_diff": d["mean_diff"],
            "ci_low": d["ci_low"],
            "ci_high": d["ci_high"],
            "tabfm_wins_%": d["a_better_pct"],
            "verdict": ("TabFM better" if decided and d["mean_diff"] < 0
                        else "TabFM worse" if decided else "within noise"),
        })

verdicts = pd.DataFrame(rows)
verdicts.to_csv(config.RESULTS / "tabfm_verdicts.csv", index=False)
verdicts

,Category,vs,mean_diff,ci_low,ci_high,tabfm_wins_%,verdict
0,Tealight Holders & Sets,Random Forest,-4.906,-10.703,0.890,66.7,within noise
1,Tealight Holders & Sets,XGBoost,-4.344,-10.277,1.590,66.7,within noise
2,Tealight Holders & Sets,Linear Regression,-8.816,-11.255,-6.377,100.0,TabFM better
3,Tealight Holders & Sets,7-Day Rolling Sum Baseline,-3.897,-12.255,4.461,66.7,within noise
4,Jewellery - Earrings,Random Forest,-4.695,-17.263,7.873,66.7,within noise
5,Jewellery - Earrings,XGBoost,0.021,-18.783,18.826,33.3,within noise
6,Jewellery - Earrings,Linear Regression,-23.806,-39.745,-7.866,100.0,TabFM better
7,Jewellery - Earrings,7-Day Rolling Sum Baseline,-47.415,-65.286,-29.544,100.0,TabFM better


## 4. Sensitivity to `n_estimators`

The comparison above used 8 forward passes instead of the library default of
32. This cell checks a single split at both settings. If the difference is
small relative to the seed-to-seed standard deviation in section 2, the
reduction did not materially handicap the model — which is the claim that
needs supporting before publishing the result.

In [ ]:
results = {}
for n_est in [8, 32]:
    mdl.enable_tabfm(n_estimators=n_est)
    t0 = time.time()
    _, vp, _ = mdl.MODELS["TabFM"](splits["train"], splits["validation"],
                                   splits["test"], features, target)
    m = evaluate_forecast(splits["validation"][target], vp)
    results[n_est] = {"val_WAPE": round(m["wape"], 2),
                      "val_R2": round(m["r2"], 3),
                      "seconds": round(time.time() - t0, 1)}

pd.DataFrame(results).T.rename_axis("n_estimators")

TabFM registered.
TabFM registered.


,val_WAPE,val_R2,seconds
n_estimators,,,
8,13.47,0.795,385.3
32,13.38,0.795,1703.1


## 5. Conclusion

*Fill in after running. Points to address:*

- Does TabFM beat the tree models, and is the interval clear of zero?
- Does it help most on the sparse category (Jewellery), as expected for a
  model with strong priors over small tables?
- Is the accuracy gain worth 6.6 GB of weights, a PyTorch dependency, and the
  per-call latency measured in section 1 — for a pipeline that must run over
  54 categories?
- The `n_estimators` reduction: supported by section 4, or a caveat?